# Capstone starter · Module 12
Python 3 and NumPy only. The capstone is Practical Lab 12, on data you can inspect. If you have no local Python environment for that lab, this notebook is the alternative route: a complete controlled experiment that you can change into your own. Either way, use the capstone template for your report.

The question below is only an example. Replace it, and write your hypothesis before you run anything.


## Part 1: Declare the protocol
Everything that defines the experiment sits in one place, and the same seeds drive both models. The hypothesis: a small CNN beats softmax regression when bars shift to new positions, because its kernels are shared across positions. It would be contradicted by an interval on the difference that sits at or below zero.


In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
PROTOCOL = dict(question='Does a small CNN beat softmax regression on shifted bars?',
                sizes=(160, 40, 200), noise=.35, train_positions=(1, 4), shifted_positions=(4, 6),
                seeds=[11, 12, 13, 14, 15], steps=300, lr=.4, primary_metric='accuracy on the shifted set')
for key, value in PROTOCOL.items(): print(f'{key:18s} {value}')
def bars(rng, n, positions):
    y = rng.integers(0, 2, n); images = np.zeros((n, 8, 8))
    for i, label in enumerate(y):
        p = int(rng.integers(*positions))
        if label == 0: images[i, p:p+2, :] = 1
        else:          images[i, :, p:p+2] = 1
    return images + rng.normal(0, PROTOCOL['noise'], images.shape), y
def softmax(z):
    shifted = z - z.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)
def nll(z, y):
    shifted = z - z.max(axis=1, keepdims=True)
    return float(np.mean(np.log(np.exp(shifted).sum(axis=1)) - shifted[np.arange(len(y)), y]))


question           Does a small CNN beat softmax regression on shifted bars?
sizes              (160, 40, 200)
noise              0.35
train_positions    (1, 4)
shifted_positions  (4, 6)
seeds              [11, 12, 13, 14, 15]
steps              300
lr                 0.4
primary_metric     accuracy on the shifted set


## Part 2: Two models, one training budget
The baseline is transparent, and the improvement changes one thing: shared local kernels with pooling. Both get the same split, seed, steps and learning rate. Training-only statistics are computed inside `run_once`, so nothing leaks.


In [2]:
def train_linear(Xtr, ytr, steps, lr):
    mu, scale = Xtr.reshape(len(Xtr), -1).mean(axis=0), Xtr.reshape(len(Xtr), -1).std(axis=0) + 1e-8
    prepare = lambda X: (X.reshape(len(X), -1) - mu) / scale
    S = prepare(Xtr); W = np.zeros((64, 2)); b = np.zeros(2); Y = np.eye(2)[ytr]
    for _ in range(steps):
        G = (softmax(S @ W + b) - Y) / len(S)
        W -= lr * S.T @ G; b -= lr * G.sum(axis=0)
    return lambda X: prepare(X) @ W + b
def windows(X): return np.lib.stride_tricks.sliding_window_view(X, (3, 3), axis=(1, 2)).reshape(len(X), 36, 9)
def cnn_forward(P, p):
    Z = P @ p['K'] + p['bc']; A = np.maximum(0, Z)
    return Z, A, A.mean(axis=1) @ p['W'] + p['b']          # global average pooling
def cnn_loss_grad(P, y, p):
    Z, A, logits = cnn_forward(P, p)
    G = (softmax(logits) - np.eye(2)[y]) / len(P)
    Gc = (G @ p['W'].T)[:, None, :] * (Z > 0) / Z.shape[1]
    return nll(logits, y), {'K': np.einsum('npk,npf->kf', P, Gc), 'bc': Gc.sum(axis=(0, 1)), 'W': A.mean(axis=1).T @ G, 'b': G.sum(axis=0)}
def cnn_init(seed, kernels=6):
    rng = np.random.default_rng(seed)
    return {'K': rng.normal(0, .3, (9, kernels)), 'bc': np.zeros(kernels), 'W': rng.normal(0, .05, (kernels, 2)), 'b': np.zeros(2)}
def train_cnn(Xtr, ytr, steps, lr, seed):
    P = windows(Xtr); p = cnn_init(seed)
    for _ in range(steps):
        _, grads = cnn_loss_grad(P, ytr, p)
        for name in p: p[name] -= lr * grads[name]
    return lambda X: cnn_forward(windows(X), p)[2]
# Trust the improvement only after its gradients pass the check from Module 05.
Xcheck, ycheck = bars(np.random.default_rng(0), 8, PROTOCOL['train_positions'])
Pcheck = windows(Xcheck); p = cnn_init(0); analytic = cnn_loss_grad(Pcheck, ycheck, p)[1]
for name, index in [('K', (1, 2)), ('bc', (3,)), ('W', (2, 1)), ('b', (0,))]:
    value = p[name][index]; h = 1e-5
    p[name][index] = value + h; plus = cnn_loss_grad(Pcheck, ycheck, p)[0]
    p[name][index] = value - h; minus = cnn_loss_grad(Pcheck, ycheck, p)[0]
    p[name][index] = value
    assert abs((plus - minus) / (2 * h) - analytic[name][index]) < 1e-6
print('CNN gradient checks passed.')
def run_once(seed):
    rng = np.random.default_rng(seed); ntr, nval, nshift = PROTOCOL['sizes']
    Xtr, ytr = bars(rng, ntr, PROTOCOL['train_positions']); Xval, yval = bars(rng, nval, PROTOCOL['train_positions'])
    Xs, ys = bars(rng, nshift, PROTOCOL['shifted_positions'])
    row = {}
    for name, model in [('baseline', train_linear(Xtr, ytr, PROTOCOL['steps'], PROTOCOL['lr'])), ('improvement', train_cnn(Xtr, ytr, PROTOCOL['steps'], PROTOCOL['lr'], seed))]:
        row[name] = dict(validation=float(np.mean(model(Xval).argmax(axis=1) == yval)), shifted=float(np.mean(model(Xs).argmax(axis=1) == ys)), shifted_nll=nll(model(Xs), ys))
    return row, model, Xs, ys
runs = {seed: run_once(seed) for seed in PROTOCOL['seeds']}
print('seed  | baseline val  shifted | improvement val  shifted')
for seed, (row, *_ ) in runs.items():
    print(f"{seed:5d} | {row['baseline']['validation']:12.3f} {row['baseline']['shifted']:8.3f} | {row['improvement']['validation']:15.3f} {row['improvement']['shifted']:8.3f}")


CNN gradient checks passed.
seed  | baseline val  shifted | improvement val  shifted
   11 |        1.000    0.180 |           1.000    0.985
   12 |        1.000    0.110 |           1.000    0.925
   13 |        1.000    0.130 |           1.000    1.000
   14 |        1.000    0.280 |           1.000    1.000
   15 |        1.000    0.120 |           1.000    1.000


## Part 3: Compare with uncertainty
The runs are paired: both models saw the same data. So compare them seed by seed. With only five seeds the interval is rough, and it describes this generator, not the world. The interval below is a bootstrap over seeds.

A score far below 50% on two balanced classes is a clue, not just a bad result: the baseline is confidently wrong, so ask what it learned instead of the orientation.


In [3]:
baseline = np.array([runs[s][0]['baseline']['shifted'] for s in PROTOCOL['seeds']])
improvement = np.array([runs[s][0]['improvement']['shifted'] for s in PROTOCOL['seeds']])
difference = improvement - baseline
draws = np.random.default_rng(0).integers(0, len(difference), (4000, len(difference)))
low, high = np.quantile(difference[draws].mean(axis=1), [.025, .975])
print('mean shifted accuracy, baseline:', baseline.mean().round(3), ' improvement:', improvement.mean().round(3))
print(f'paired difference {difference.mean():.3f}, bootstrap interval {low:.3f} to {high:.3f}, from {len(difference)} seeds')
print('Hypothesis supported on this generator?', bool(low > 0))
assert len(difference) == len(PROTOCOL['seeds']) and np.all((0 <= baseline) & (baseline <= 1))


mean shifted accuracy, baseline: 0.164  improvement: 0.982
paired difference 0.818, bootstrap interval 0.769 to 0.863, from 5 seeds
Hypothesis supported on this generator? True


## Part 4: Audit the failures
A metric says how often, and failures say how. Pull out the most confident mistakes and look at them. Write one explanation you can test, and one you cannot yet separate from chance.


In [4]:
def show(image):
    shades = ' .:-=+*#%@'
    for line in np.clip(image, 0, .99):
        print(''.join(shades[int(v * 10)] * 2 for v in line))
row, model, Xs, ys = runs[PROTOCOL['seeds'][0]]
probabilities = softmax(model(Xs)); predicted = probabilities.argmax(axis=1); confidence = probabilities.max(axis=1)
wrong = np.where(predicted != ys)[0]
print('errors:', len(wrong), 'of', len(ys))
for index in wrong[np.argsort(confidence[wrong])[::-1]][:3]:
    print(f'\nimage {index}: true {"vertical" if ys[index] else "horizontal"}, predicted {"vertical" if predicted[index] else "horizontal"}, confidence {confidence[index]:.3f}')
    show(Xs[index])
print('\nNext: state the population this evidence supports, and what would change your conclusion.')


errors: 3 of 200

image 104: true horizontal, predicted vertical, confidence 0.633
--        ..::  
  --    **      
==--    ::    --
    ::  **++    
--::::  ++      
@@%%@@++++@@**++
@@@@@@@@@@@@@@##
                

image 53: true vertical, predicted horizontal, confidence 0.516
::        @@@@++
....::    @@@@  
      ++++##@@  
--..    --@@==++
  ::        @@::
  **++==  @@@@++
          **@@  
        --%%@@  

image 165: true vertical, predicted horizontal, confidence 0.504
    --  --@@##==
        ==++@@  
..  ::--==@@@@..
==  ..@@  ==@@##
  ::::    **##..
  --  ++  @@@@..
--  ==    @@@@  
++    ==  ==%%%%

Next: state the population this evidence supports, and what would change your conclusion.


## Make it your capstone
Replace the question, change one factor at a time, and keep the protocol cell honest: if you change a setting after seeing a result, record that you did. Fill in the capstone template with your question, data protocol, shapes, one derivative, the comparison table, three failures and your limitations. A candid negative result is a successful capstone.
